# Course notebook — Training, Serving, and AI Operations

Understand model adaptation, efficient inference, deployment, observability, release, and lifecycle operations.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · The supplier assistant that learned last year’s policy

Wei operates an assistant for a purchasing team. The team wants a consistent approval format and accurate current pricing guidance, but contract terms can change weekly. A pilot must also survive peak workload on order day with an identifiable release and rollback decision.

The workplace scenario is fictional.


## Model adaptation and post-training


### Lecture 01 · Choosing Adaptation

Diagnose whether a requirement needs prompting, retrieval, tools, fine-tuning, continued pretraining, or a different model.

#### Separate changing facts from persistent behavior

The assistant skips a required approval field and also gives an outdated contract amount.

A structured constraint or adaptation can improve the recurring format. The current amount belongs in versioned retrieval. Test the two behaviors separately to avoid paying to retrain the model whenever a contract changes.

#### Read the mechanism

**behavior versus knowledge**

Behavior changes how the model acts—tone, format, policy—while knowledge is factual content. RAG adds knowledge; fine-tuning often shifts behavior.

Model knows refunds exist but needs SFT to always ask order ID first—that is behavior.

**prompting**

Prompting steers model behavior at inference via instructions and examples without weight updates. It is the fastest iteration path when context fits.

Adding 'cite sources' instruction improves citation rate without retraining.

**RAG**

Retrieval-augmented generation retrieves external evidence at query time and conditions generation on it.

HR assistant retrieves current travel policy and refuses when no supporting document exists.

**fine-tuning**

Fine-tuning adapts pretrained weights with supervised or preference data when prompts and RAG cannot stabilize behavior. It trades generality and ops simplicity for targeted changes.

Support tone and escalation policy may need SFT when prompts drift across thousands of ticket types.

**model selection**

Model selection matches capabilities, cost, latency, license, and risk to task requirements—not brand prestige.

Small model handles classification; large model only for complex reasoning slice.

#### Worked example

**Situation:** A service must route requests across models while controlling cost and retaining rollback. Team debates prompt vs RAG vs fine-tune for tone compliance.

**Baseline:** Fine-tune immediately for every behavioral tweak.

**Application:** Decision table for ten scenarios separating knowledge gaps (RAG), style/behavior (prompt/fine-tune), tool needs, latency budgets; pick smallest intervention at correct layer.

**Test cases:** (1) Normal: update policy answer via retrieval. (2) Boundary: consistent refusal tone. (3) Adversarial: fine-tune on contaminated eval examples.

**Measurement:** Task success per intervention, operational cost, rollback complexity score.

**Design question:** Which scenario incorrectly defaults to fine-tuning when retrieval would suffice?

**Predict before running.** Choose the smallest intervention at the correct system layer. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 11.1: Choosing Adaptation"""

CHAPTER = "11.1"
print("chapter hook:", CHAPTER)
scenarios = [
    ("new policy fact", "RAG"),
    ("consistent tone", "prompt/SFT"),
    ("live database count", "tool"),
]
for need, fix in scenarios:
    print({"need": need, "intervention": fix})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Create a decision table for ten adaptation scenarios.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why choose smallest intervention at correct layer?
2. When does RAG beat fine-tuning for knowledge?
3. What adaptation baseline always retrains?

<details><summary>Check your reasoning</summary>

Q1: Avoids unnecessary cost and rigidity. Q2: Facts change frequently and need citations. Q3: Full fine-tune for every FAQ update.

</details>


### Lecture 02 · Post-Training Methods

Understand supervised fine-tuning, LoRA, QLoRA, preference data, RLHF, DPO, distillation, and model merging.

#### Adapting the model can improve the wrong thing

A tuned version matches the training examples more closely. Reviewers notice it now sounds certain when an approval is missing.

Hold out realistic new requests and include abstentions in the behavioral criteria. Compare the base and adapted versions on formatting and uncertainty independently before interpreting a loss improvement.

#### Read the mechanism

**SFT**

Supervised fine-tuning trains on input–output pairs to imitate desired behaviors on similar tasks.

SFT on 5k support replies teaches consistent empathy and escalation triggers.

**LoRA**

LoRA fine-tunes low-rank adapter matrices in attention layers, reducing trainable parameters versus full fine-tuning.

7B model with LoRA learns domain tone on one GPU while base weights stay frozen.

**QLoRA**

QLoRA combines quantization of base weights with LoRA adapters for fine-tuning on consumer GPUs.

Fine-tune 13B on single 24GB card using 4-bit base plus LoRA adapters.

**DPO**

Direct Preference Optimization aligns models from pairwise preferences without explicit reward model training.

Prefer concise accurate answers over verbose wrong ones via DPO preference pairs.

**distillation**

Distillation trains smaller student models to mimic larger teachers, trading capability for cost and speed.

Student classifier matches teacher on 95% of eval at 5× lower latency.

#### Worked example

**Situation:** Support assistant needs tighter adherence to escalation phrasing; base instruct model drifts on edge cases.

**Baseline:** Longer prompts only—context cost rises, drift remains.

**Application:** LoRA fine-tune on curated escalation dialogs, compare SFT vs DPO if preference data exists, evaluate held-out behavioral cases, document merge/deploy plan.

**Test cases:** (1) Normal: standard escalation wording. (2) Boundary: rare dual-escalation case. (3) Adversarial: overfitting to training templates hurts novel incidents.

**Measurement:** Behavioral eval pass rate, general capability regression suite, training GPU cost.

**Design question:** How much general capability regression is acceptable for a 5-point slice gain?

**Predict before running.** Adaptation trades generality and operational simplicity for targeted behavior. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 11.2: Post-Training Methods"""

CHAPTER = "11.2"
print("chapter hook:", CHAPTER)
methods = {"prompt": 0.82, "SFT": 0.91, "DPO": 0.93}
cost = {"prompt": 1, "SFT": 4, "DPO": 6}
target = 0.90
choice = min((m for m, s in methods.items() if s >= target), key=lambda m: cost[m])
print({"method": choice, "score": methods[choice]})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Fine-tune a small model and evaluate held-out behavior.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What trade does adaptation make against generality?
2. When prefer DPO over SFT?
3. What post-training baseline is prompt-only?

<details><summary>Check your reasoning</summary>

Q1: Targeted behavior vs broader capability and ops complexity. Q2: Clear preference pairs on style/safety judgments. Q3: Zero-shot instruct with no weight updates.

</details>


**Section checkpoint.** Specify a behavior where prompt changes are a reasonable baseline before considering an adapter.


## Datasets and inference infrastructure


### Lecture 03 · Dataset Engineering

Curate, label, deduplicate, filter, balance, version, document, and protect training and evaluation data.

#### Remove the answer from the training window

The submitted record contains both a draft and the later reviewer correction. Splitting by row puts variants of one supplier request into both training and evaluation.

Group by supplier and workflow before splitting. Version the transformation that decides which fields would have existed at inference time. Track excluded examples so the next run can reproduce the data decision.

#### Read the mechanism

**data curation**

Data curation selects, cleans, and balances training examples for quality over quantity. Garbage data teaches garbage behavior.

Removing toxic and duplicate examples improves fine-tune safety more than doubling raw size.

**synthetic data**

Synthetic data generates training examples via models or rules—useful when real data is scarce but risks model collapse if overused.

GPT generates varied phrasings of intent labels to augment small classifier set.

**deduplication**

Deduplication removes near-duplicate training examples that inflate metrics and memorization.

Duplicate FAQ pairs in SFT data cause verbatim regurgitation in deployment.

**contamination**

Contamination occurs when eval examples leak into training data, inflating benchmark scores.

Near-duplicate test questions in fine-tune set invalidate held-out claims.

**data cards**

Data cards document dataset sources, collection, demographics, limitations, and recommended uses—parallel to model cards.

Fine-tune data card lists languages, date range, PII handling, and opt-out process.

#### Worked example

**Situation:** Fine-tune dataset assembled from historical chats; legal discovers eval tickets leaked into training.

**Baseline:** Dump all logs into JSONL without dedup or contamination checks.

**Application:** Curate splits, deduplicate near-duplicates, filter PII, version dataset, write data card, run contamination scan against eval sets, document synthetic augmentation choices.

**Test cases:** (1) Normal: clean curated set. (2) Boundary: synthetic examples labeled as such. (3) Adversarial: near-duplicate paraphrases of eval cases in train.

**Measurement:** Contamination hits (target zero), dedup ratio, label error rate on audit sample.

**Design question:** Which check catches eval leakage that random splitting misses?

**Predict before running.** Data design is model behavior design. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 11.3: Dataset Engineering"""

CHAPTER = "11.3"
print("chapter hook:", CHAPTER)
train = {"case-101", "case-102", "case-103"}
eval = {"case-103", "case-200"}
leak = train & eval
print({"leaked_ids": sorted(leak)})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Create a data card and contamination check for a small dataset.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is data design model behavior design?
2. How does deduplication affect generalization estimates?
3. What dataset baseline uses random split only?

<details><summary>Check your reasoning</summary>

Q1: Labels, balance, and contamination define what model learns. Q2: Duplicates inflate train metrics vs honest generalization. Q3: Train/test split without near-dup or ID checks.

</details>


### Lecture 04 · Inference Infrastructure

Connect accelerators, memory, quantization, model formats, servers, batching, streaming, caches, and speculative decoding.

#### Latency belongs to the workload, not the badge

The model serves short requests quickly but slows when 30 long contract excerpts arrive together. The queue grows during peak order entry.

Separate arrival rates, input length, generation work, and wait time. Compare a small candidate with batching or input limits on a representative mix before assuming a single hardware change solves the bottleneck.

#### Read the mechanism

**GPUs**

GPUs accelerate matrix operations for training and inference; memory capacity limits model size and batch.

80GB GPU runs 70B quantized; 24GB fits 7B fine-tune with QLoRA.

**quantization**

Quantization reduces weight precision—INT8, INT4—to cut memory and increase throughput with small quality trade-offs.

AWQ 4-bit model runs 2× faster with <1 point eval drop on some tasks.

**vLLM**

vLLM is a high-throughput inference server using PagedAttention for efficient KV cache memory management.

vLLM serves Llama-8B at higher concurrent requests than naive HuggingFace pipeline.

**batching**

Batching groups requests to amortize GPU kernel overhead, improving throughput at possible latency cost. Continuous batching in servers interleaves sequences of different lengths.

Batch size 32 may double throughput versus batch 1 but increase p95 latency for short prompts.

**KV cache**

The KV cache stores key and value tensors for prior tokens during autoregressive decoding, avoiding recomputation of the prefix. Memory grows linearly with context length.

Streaming chat reuses cached states for system prompt and prior turns, cutting latency after the first token.

#### Worked example

**Situation:** Self-hosted inference must serve onboarding assistant peaks; latency spikes when concurrency jumps.

**Baseline:** Single-process model server batch size 1.

**Application:** Load-test at concurrency 1/4/8/16, measure tokens/sec and p95 latency, explore quantization trade-offs, estimate KV cache memory from context length distribution.

**Test cases:** (1) Normal: steady 4 concurrent. (2) Boundary: context exactly at cache limit. (3) Adversarial: all requests unique prefixes—cache useless.

**Measurement:** Throughput curve, p95 latency, GPU memory headroom, $/1M tokens.

**Design question:** At what concurrency does queueing dominate over compute?

**Predict before running.** Inference performance is a queueing and memory problem as much as a model problem. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 11.4: Inference Infrastructure"""

CHAPTER = "11.4"
print("chapter hook:", CHAPTER)
batch_sizes = [1, 4, 8]
for b in batch_sizes:
    throughput = b / (1 + 0.1 * (b - 1))
    print(f"batch={b} relative_throughput={throughput:.2f}")
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Load-test a local model at several concurrency levels.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is inference a queueing and memory problem?
2. When does KV cache stop helping?
3. What infra baseline ignores batching?

<details><summary>Check your reasoning</summary>

Q1: Requests wait in queues; memory bounds batch and context. Q2: Unique prefixes every request—no reuse. Q3: Serial single-request server at scale.

</details>


**Section checkpoint.** Describe a held-out slice and a serving constraint. Explain which one affects training versus deployment.


## Deployment and operations


### Lecture 05 · Deployment and Routing

Design containers, serverless endpoints, Kubernetes, autoscaling, routing, fallbacks, regional placement, and disaster recovery.

#### Make fallback independent enough to help

The primary candidate times out during payroll. Its fallback uses the same overloaded service and fails during the same window.

Test a degraded mode that can run under different constraints, even if it only creates a review task. Verify that the user sees the status and knows who owns the next action.

#### Read the mechanism

**containers**

Containers package model servers with dependencies for reproducible deployment across environments.

Docker image pins CUDA, Python, and model weights hash for prod inference.

**autoscaling**

Autoscaling adjusts inference replica count based on CPU, GPU utilization, or queue depth.

Scale GPU pods from 2 to 10 when p95 queue wait exceeds 500ms.

**model routing**

Model routing directs requests to appropriate models by task, risk, cost, or latency policy.

Regex on ticket category routes billing to fine-tuned small model, general to large.

**fallbacks**

Fallbacks switch to alternate models, cached answers, or human handoff when primary path fails.

If primary API 503, serve smaller local model with degraded-quality banner.

**resilience**

Resilience designs for partial failure—retries, circuit breakers, multi-region—without total service loss.

Circuit breaker stops calling failing embedding API after 50% errors, uses lexical only.

#### Worked example

**Situation:** Team chooses between hosted API and self-hosted Kubernetes for inference with rollback requirements.

**Baseline:** Direct latest-model endpoint in production code.

**Application:** Write ADR comparing containers vs serverless vs hosted: control, cost at forecast load, latency, failover, regional placement, disaster recovery, model routing and fallbacks.

**Test cases:** (1) Normal: primary model available. (2) Boundary: provider outage triggers fallback. (3) Adversarial: silent provider behavior change without version pin.

**Measurement:** Failover time, cost at 1M req/mo, ops burden score (1–5).

**Design question:** What requirement forces self-host despite higher ops burden?

**Predict before running.** Deployment choices allocate control, cost, latency, and operational burden. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 11.5: Deployment and Routing"""

CHAPTER = "11.5"
print("chapter hook:", CHAPTER)
options = {"hosted": {"control": 2, "cost": 3}, "self": {"control": 5, "cost": 4}}
need = "data residency strict"
choice = "self" if "residency" in need else "hosted"
print({"choice": choice, **options[choice]})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Write a deployment ADR comparing hosted and self-hosted inference.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. How do deployment choices allocate control and cost?
2. When are fallbacks mandatory?
3. What deployment baseline has no ADR?

<details><summary>Check your reasoning</summary>

Q1: Hosted trades control for speed; self-host inverts. Q2: Any production user-facing path with SLA needs fallback model/route. Q3: First vendor picked ad hoc.

</details>


### Lecture 06 · LLMOps

Version prompts, models, data, and evals; trace requests; monitor quality and cost; canary, roll back, and respond to incidents.

#### Ask where the regression entered the pipeline

Operations sees fewer correct citations after a release. The checkpoints, quantization, and document index were all updated together.

Record all relevant versions and change them separately where possible. Compare slices and retrieve-stage evidence before restoring the right artifact or rerouting to the prior configuration.

#### Read the mechanism

**tracing**

Tracing records spans for retrieval, model calls, tools, and validation with correlation IDs across services.

OpenTelemetry trace shows 400ms in reranker, 1.2s in LLM for slow request diagnosis.

**versioning**

Versioning tracks prompts, models, indexes, and eval suites so changes are attributable and reversible.

Prod trace includes prompt v3.1, model llama-3-8b-q4, index 2024-06-01.

**continuous evaluation**

Continuous evaluation runs production or shadow traffic against eval suites to detect drift post-release.

Nightly job scores 500 sampled prod queries with LLM judge against rubric.

**canaries**

Canaries route small traffic percentage to new versions before full rollout.

5% traffic to new embedding index for 24h comparing recall and latency.

**FinOps**

FinOps tracks and optimizes AI spend—tokens, GPU hours, API fees—against business value.

Dashboard shows cost per successful ticket deflection by model route.

#### Worked example

**Situation:** Production traces show occasional wrong answers after retrieval provider blips; finance wants cost attribution per feature.

**Baseline:** Logs only final response text.

**Application:** Instrument full request trace (model version, retrieval latency, validation outcome), inject failure drills for provider/retrieval/validation, canary releases with automatic rollback on eval regression.

**Test cases:** (1) Normal: full trace captured. (2) Boundary: partial trace when tool times out. (3) Adversarial: validation bypass bug ships in canary.

**Measurement:** MTTR on injected failures, trace completeness %, cost per successful request by stage.

**Design question:** Which signal triggers rollback fastest with lowest false positives?

**Predict before running.** Every production change needs evidence, observability, and a reversible release path. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 11.6: LLMOps"""

CHAPTER = "11.6"
print("chapter hook:", CHAPTER)
canary = {"success_rate": 0.79, "baseline": 0.85, "threshold": -0.03}
delta = canary["success_rate"] - canary["baseline"]
action = "rollback" if delta < canary["threshold"] else "promote"
print({"delta": round(delta, 3), "action": action})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Instrument a request and inject provider, retrieval, and validation failures.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why must production changes be reversible with evidence?
2. What does tracing enable beyond debugging?
3. What LLMOps baseline lacks versioning?

<details><summary>Check your reasoning</summary>

Q1: Regressions happen; rollback limits blast radius. Q2: Cost attribution, stage failures, audit. Q3: Deploy latest prompt/model with no canary.

</details>


**Section checkpoint.** Compare routing choices and write a rollback trigger in terms of latency and quality.


## Course workshop

Compare the router's decisions for a simple task and a high-risk task. Draft a separate held-out evaluation for any future fine-tune.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Policy-based model routing with risk and fallback controls."""
MODELS = {
    "small": {"quality": 0.75, "cost": 1},
    "large": {"quality": 0.93, "cost": 8},
}


def route(task):
    if task["risk"] == "high" or task["complexity"] > 0.7: return "large"
    return "small"


for task in ({"risk": "low", "complexity": .2}, {"risk": "high", "complexity": .3}):
    model = route(task)
    print(task, "->", model, MODELS[model])


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
assert route({"complexity": 0.2, "risk": "low"}) != route({"complexity": 0.2, "risk": "high"})
print('Workshop checks passed')


## Applied exercise for Lecture 02 — Train a rank-one adapter and evaluate held-out inputs

A frozen two-output model returns the input vector. Fit two small vectors so their outer product adds a rank-one correction. Compare base and adapted error on held-out inputs, then reduce the training steps and rerun. This exercise uses a tiny linear model to make the update visible. A transformer has many more weights and training choices.


In [ ]:
training = [(1.0, 0.0), (0.0, 1.0), (-1.0, -1.0), (2.0, 1.0)]
holdout = [(2.0, -1.0), (-2.0, 3.0)]


def target(point):
    x, y = point
    signal = 2 * x - y
    return [x + signal, y - 0.5 * signal]


def base(point):
    return list(point)


a = [0.1, -0.1]
b = [0.0, 0.0]


def adapted(point):
    hidden = sum(weight * value for weight, value in zip(a, point))
    return [value + weight * hidden for value, weight in zip(base(point), b)]


for _ in range(200):
    for point in training:
        residual = [prediction - wanted for prediction, wanted in zip(adapted(point), target(point))]
        hidden = sum(weight * value for weight, value in zip(a, point))
        gradient_hidden = sum(error * weight for error, weight in zip(residual, b))
        a = [weight - 0.04 * gradient_hidden * value for weight, value in zip(a, point)]
        b = [weight - 0.04 * error * hidden for weight, error in zip(b, residual)]


def squared_error(predict):
    return sum(
        (observed - expected) ** 2
        for point in holdout
        for observed, expected in zip(predict(point), target(point))
    )


base_error = squared_error(base)
adapted_error = squared_error(adapted)
print(f"Held-out squared error: base={base_error:.3f}, adapted={adapted_error:.3f}")
assert adapted_error < 0.1 * base_error


## Course project

Adapt and serve a small model, benchmark it, instrument it, and release it with rollback evidence.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
